### PHASE 5 — Friday | Crisis Timeline + Time Intelligence

---

`Business Request from CEO:`
"When did this crisis start? Was it sudden or gradual? I need Q4 2023 default trends for the board meeting next week."

`Mindset Unlock:`
The finale. Today's job is not just to count, but to predict. A 5% default rate that has stayed 5% for six months is stable. A 5% default rate that was 1% last month is a breakout. Velocity, the rate of change over time, is the most important metric for any executive. You are looking for the exact moment the EduFin portfolio lost its health.

`CEO'S EXECUTIVE BRIEF — "When did this start, and how fast is it moving?"` 

The Board of Directors is meeting on Monday. They need to know the 'Genesis Month' of this default surge and the current 'Velocity' (month-over-month growth rate). Your final report must identify the trend, show the acceleration, and synthesize insights from all previous phases into one coherent narrative.


Start Spark Session

---

In [1]:
## Import dependencies and create Spark session
import time
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/30 13:59:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Global Variables

---

In [2]:
N = 30 # Number of rows to show in results
schema = "edufin_small"  # edufin_small or edufin_national

Load the data

---

In [ ]:
## Load the datasets and create a temporary views for SQL queries
loans = spark.read.csv(f"../datasets/{schema}/loans.csv", header=True)
defaults_collections = spark.read.csv("../datasets/defaults_collections.csv", header=True)

loans.createOrReplaceTempView("loans")
defaults_collections.createOrReplaceTempView("defaults_collections")

loans.show(n=10)
defaults_collections.show(n=10)

+-------+-----------+--------------+-----------+-----------+-------------+------------------+----------------+-----------------+-------------+-----------+--------------------+
|loan_id|customer_id|institution_id|loan_amount|loan_status|interest_rate|loan_tenure_months|application_date|disbursement_date|maturity_date| emi_amount|     purpose_of_loan|
+-------+-----------+--------------+-----------+-----------+-------------+------------------+----------------+-----------------+-------------+-----------+--------------------+
|      1|       2440|          2954| 190607.125|  Defaulted|  11.39000034|                84|      08-12-2021|       23-12-2021|   16-11-2028|3302.540039|     Living Expenses|
|      2|       2440|          4741| 425798.375|     Active|  14.43999958|                48|      01-01-2022|       11-01-2022|   21-12-2025|11728.73047|Course Fees + Living|
|      3|       2440|           902|  318341.25|  Defaulted|  11.64999962|                96|      06-03-2023|       12-

Query 5A (BRD): Monthly Disbursement Trends

---

- Calculate: Total loans and amount disbursed per month (last 24 months).
- Business Purpose: Visualize growth trajectory.

In [4]:
query = """
SELECT 
    MIN(try_to_date(disbursement_date, 'dd-MM-yyyy')) AS earliest_date,
    MAX(try_to_date(disbursement_date, 'dd-MM-yyyy')) AS latest_date,
    COUNT(*) AS total_records
FROM loans
WHERE try_to_date(disbursement_date, 'dd-MM-yyyy') >= ADD_MONTHS(CURRENT_DATE(), -24)
"""
spark.sql(query).show(truncate=False)

+-------------+-----------+-------------+
|earliest_date|latest_date|total_records|
+-------------+-----------+-------------+
|2024-05-30   |2025-07-03 |1396         |
+-------------+-----------+-------------+



In [5]:
query = """
WITH monthly_disbursements AS (
    SELECT 
        DATE_FORMAT(
            COALESCE(
                try_to_date(disbursement_date, 'dd-MM-yyyy'),
                try_to_date(disbursement_date, 'yyyy-MM-dd')
            ), 
            'yyyy-MM'
        ) AS disbursement_month,
        COUNT(*) AS total_loans_disbursed,
        CASE 
            WHEN SUM(loan_amount) > 10000000.0 THEN CONCAT(ROUND(SUM(loan_amount) / 10000000.0, 2), ' Cr')
            ELSE CONCAT(ROUND(SUM(loan_amount) / 100000.0, 2), ' L') 
        END AS total_amount_disbursed
    FROM loans
    GROUP BY DATE_FORMAT(
        COALESCE(
            try_to_date(disbursement_date, 'dd-MM-yyyy'),
            try_to_date(disbursement_date, 'yyyy-MM-dd')
        ), 
        'yyyy-MM'
    )
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY disbursement_month) AS SN,
    disbursement_month AS `Disbursement Month`,
    total_loans_disbursed AS `Loans Disbursed`,
    total_amount_disbursed AS `Amount Disbursed`
FROM monthly_disbursements
ORDER BY disbursement_month;
"""
start = time.perf_counter()
spark.sql(query).show(n=25, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+---+------------------+---------------+----------------+
|SN |Disbursement Month|Loans Disbursed|Amount Disbursed|
+---+------------------+---------------+----------------+
|1  |2021-08           |44             |1.94 Cr         |
|2  |2021-09           |103            |3.98 Cr         |
|3  |2021-10           |111            |4.61 Cr         |
|4  |2021-11           |105            |4.35 Cr         |
|5  |2021-12           |104            |3.88 Cr         |
|6  |2022-01           |122            |5.23 Cr         |
|7  |2022-02           |82             |3.32 Cr         |
|8  |2022-03           |113            |4.42 Cr         |
|9  |2022-04           |96             |4.19 Cr         |
|10 |2022-05           |104            |4.42 Cr         |
|11 |2022-06           |106            |4.27 Cr         |
|12 |2022-07           |114            |4.52 Cr         |
|13 |2022-08           |121            |4.77 Cr         |
|14 |2022-09           |125            |5.32 Cr         |
|15 |2022-10  

STEP 5A (Workbook) — Open Metric: "Genesis Month" Discovery

---

What you're doing: The Board wants to know the EXACT moment we lost control. You must define what "Out of Control" means quantitatively (e.g., Velocity > 50%).

Your 'Genesis Month' Definition Logic:

*e.g., The month where (defaults / issuances) growth significantly deviates from the annual baseline. Propose your threshold.*

-> *Genesis Month = The first month where the default rate spiked by >50% compared to the previous month*

Your SQL Implementation:

In [13]:
query = """
WITH base_date AS (
    SELECT 
        YEAR(TO_DATE(disbursement_date, 'dd-MM-yyyy')) as disbursement_year,
        CASE 
            WHEN MONTH(TO_DATE(disbursement_date, 'dd-MM-yyyy')) <= 3  THEN 'Q1' 
            WHEN MONTH(TO_DATE(disbursement_date, 'dd-MM-yyyy')) <= 6  THEN 'Q2'
            WHEN MONTH(TO_DATE(disbursement_date, 'dd-MM-yyyy')) <= 9  THEN 'Q3'
            WHEN MONTH(TO_DATE(disbursement_date, 'dd-MM-yyyy')) <= 12 THEN 'Q4' 
        END AS quarter_name,
        loan_amount,
        loan_id,
        customer_id
    FROM loans
)
SELECT 
    disbursement_year AS `Disbursement Year`, 
    CONCAT(disbursement_year, '-', quarter_name) AS `Disbursement Quarter`,
    CONCAT(ROUND(SUM(loan_amount)/10000000, 2), ' Cr') AS `Loan Amount`,
    COUNT(loan_id) AS `Total Loans`,
    COUNT(DISTINCT customer_id) AS `Total Customers`,
    CONCAT(ROUND(AVG(loan_amount)/100000, 2), ' L') AS `Average Loan Size (in Lakh)`
FROM base_date
GROUP BY disbursement_year, quarter_name
ORDER BY disbursement_year, quarter_name;
"""
start = time.perf_counter()
spark.sql(query).show(n=100, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-----------------+--------------------+-----------+-----------+---------------+---------------------------+
|Disbursement Year|Disbursement Quarter|Loan Amount|Total Loans|Total Customers|Average Loan Size (in Lakh)|
+-----------------+--------------------+-----------+-----------+---------------+---------------------------+
|2021             |2021-Q3             |5.93 Cr    |147        |144            |4.03 L                     |
|2021             |2021-Q4             |12.84 Cr   |320        |308            |4.01 L                     |
|2022             |2022-Q1             |12.97 Cr   |317        |307            |4.09 L                     |
|2022             |2022-Q2             |12.89 Cr   |306        |292            |4.21 L                     |
|2022             |2022-Q3             |14.61 Cr   |360        |343            |4.06 L                     |
|2022             |2022-Q4             |13.34 Cr   |330        |316            |4.04 L                     |
|2023             |

Query 5B (BRD): Tracking Defaults by Vintage

---

- Calculate: How many loans from each month's cohort eventually defaulted.
- Business Purpose: Cohort analysis to pinpoint bad vintage months.

In [14]:
query = """
SELECT 
    DATE_TRUNC('month', try_to_date(application_date, 'dd-MM-yyyy')) AS vintage_month,
    COUNT(*) AS total_loans,
    SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) AS defaults,
    ROUND(100.0 * SUM(CASE WHEN loan_status = 'Defaulted' THEN 1 ELSE 0 END) / COUNT(*), 2) AS default_rate_pct
FROM loans
GROUP BY DATE_TRUNC('month', try_to_date(application_date, 'dd-MM-yyyy'))
ORDER BY vintage_month
"""

start = time.perf_counter()
result_df = spark.sql(query)
result_df.show(n=N, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-------------------+-----------+--------+----------------+
|vintage_month      |total_loans|defaults|default_rate_pct|
+-------------------+-----------+--------+----------------+
|2021-07-01 00:00:00|24         |0       |0.00            |
|2021-08-01 00:00:00|115        |13      |11.30           |
|2021-09-01 00:00:00|102        |7       |6.86            |
|2021-10-01 00:00:00|116        |13      |11.21           |
|2021-11-01 00:00:00|95         |13      |13.68           |
|2021-12-01 00:00:00|126        |7       |5.56            |
|2022-01-01 00:00:00|99         |11      |11.11           |
|2022-02-01 00:00:00|105        |11      |10.48           |
|2022-03-01 00:00:00|94         |7       |7.45            |
|2022-04-01 00:00:00|97         |12      |12.37           |
|2022-05-01 00:00:00|107        |10      |9.35            |
|2022-06-01 00:00:00|122        |18      |14.75           |
|2022-07-01 00:00:00|103        |14      |13.59           |
|2022-08-01 00:00:00|131        |15     

STEP 5B (Workbook) — Monthly Default Trend

---

What you're doing: Count how many loans defaulted each month. Look for the first month where defaults exceeded 50, then 100, then 200. This is the raw timeline of the crisis.

> When did the count first double in a single month? That is your 'Trigger Point'.

In [19]:
query = """
SELECT
    YEAR(TO_DATE(disbursement_date, 'dd-MM-yyyy')) AS disbursement_year,
    CONCAT(
        YEAR(TO_DATE(disbursement_date, 'dd-MM-yyyy')), '-Q',
        CASE 
            WHEN MONTH(TO_DATE(disbursement_date, 'dd-MM-yyyy')) <= 3  THEN 1
            WHEN MONTH(TO_DATE(disbursement_date, 'dd-MM-yyyy')) <= 6  THEN 2
            WHEN MONTH(TO_DATE(disbursement_date, 'dd-MM-yyyy')) <= 9  THEN 3
            WHEN MONTH(TO_DATE(disbursement_date, 'dd-MM-yyyy')) <= 12 THEN 4
        END
    ) AS quarter_name,
    ROUND(SUM(loan_amount) / 10000000, 2) AS loan_amount_cr,
    COUNT(loans.loan_id) AS total_loans,
    COUNT(DISTINCT loans.customer_id) AS total_customers,
    ROUND(
        COUNT(CASE WHEN loan_status = 'Defaulted' THEN 1 END) * 100.0
        / COUNT(*), 2
    ) AS default_rate,
    ROUND(
        AVG(CASE 
            WHEN loan_status = 'Defaulted' 
            THEN MONTHS_BETWEEN(
                TO_DATE(defaults_collections.default_date, 'dd-MM-yyyy'),
                TO_DATE(disbursement_date, 'dd-MM-yyyy')
            )
        END), 2
    ) AS months_to_default,
    COUNT(CASE WHEN loan_status = 'Defaulted' THEN 1 END) AS defaulted_loans
FROM loans
LEFT JOIN defaults_collections
    ON loans.loan_id = defaults_collections.loan_id
GROUP BY
    YEAR(TO_DATE(disbursement_date, 'dd-MM-yyyy')),
    CASE 
        WHEN MONTH(TO_DATE(disbursement_date, 'dd-MM-yyyy')) <= 3  THEN 1
        WHEN MONTH(TO_DATE(disbursement_date, 'dd-MM-yyyy')) <= 6  THEN 2
        WHEN MONTH(TO_DATE(disbursement_date, 'dd-MM-yyyy')) <= 9  THEN 3
        WHEN MONTH(TO_DATE(disbursement_date, 'dd-MM-yyyy')) <= 12 THEN 4
    END
ORDER BY disbursement_year, quarter_name;
"""

start = time.perf_counter()
result_df = spark.sql(query)
result_df.show(n=N, truncate=False)
print(f"Time: {(time.perf_counter() - start):.3f} seconds")

+-----------------+------------+--------------+-----------+---------------+------------+-----------------+---------------+
|disbursement_year|quarter_name|loan_amount_cr|total_loans|total_customers|default_rate|months_to_default|defaulted_loans|
+-----------------+------------+--------------+-----------+---------------+------------+-----------------+---------------+
|2021             |2021-Q3     |5.93          |147        |144            |10.20       |24.4             |15             |
|2021             |2021-Q4     |12.84         |320        |308            |10.31       |21.88            |33             |
|2022             |2022-Q1     |12.97         |317        |307            |8.52        |20.89            |27             |
|2022             |2022-Q2     |12.89         |306        |292            |10.46       |24.13            |32             |
|2022             |2022-Q3     |14.61         |360        |343            |12.78       |20.35            |46             |
|2022           

Query 5C (BRD): Monthly Default Rate Trends

---

- Calculate: Default Rate % for each month.
- Business Purpose: The core "Crisis Timeline" metric.

STEP 5C (Workbook) — Default Velocity (Month-over-Month Growth)

---

What you're doing: Use LAG() or a self-join to compare this month's defaults to last month's defaults. Calculate the percentage growth. This 'Velocity' metric is what captures executive attention.

> Is the velocity increasing, decreasing, or plateauing in the most recent two months? This determines if the crisis is 'out of control' or 'stabilizing'.

Query 5D (BRD): Cumulative Financial Impact

---

- Calculate: Rolling total of losses over time.
- Business Purpose: See how the financial hole grew.

STEP 5D (Workbook) — Lag Time (Origination to Default)

---

What you're doing: Calculate the average number of days between the loan start date and the default date. This shows 'Survival Time'. If survival time is dropping, it means new loans are defaulting faster than old ones.

> Note the lowest average survival time. (e.g., 'In August, loans were defaulting in just X days on average').

Query 5E (BRD): Crisis Timeline Dashboard

---

- Calculate: Month-by-month crisis heat map.
- Business Purpose: Pinpoint the exact "Point of Failure" (month where defaults spiked).
- Classifications: CRISIS MONTH (>15%), WARNING MONTH (10-15%).
- REQUIRED Text Summary: Identify the exact month the crisis began and whether it is currently stabilizing or worsening.

STEP 5E (Workbook) — Final Strategic Synthesis (Executive Dashboard)

---

What you're doing: Combine your core findings from all phases into a single, high-stakes table. This is the summary the Board of Directors will use to decide if they should fire the CEO or double down on collections.


> Read your 'Analyst Notes' (status labels). Do they tell a clear story of deterioration across the phases? 